# LoRA fine-tuning Qwen3.5-4B на Text2SQL

Полный пайплайн: install → load model → train LoRA → merge → GGUF → download.

**Окружение:** Google Colab, Tesla T4 (16 GB VRAM) — бесплатный tier.

**Входные данные** (загрузить через `Files` ВРУЧНУЮ или подключить Google Drive):
- `train.jsonl` — выход `scripts/build_finetune_dataset.py`
- `val.jsonl` — то же

**Учебные нюансы по ходу:**
- Почему unsloth, а не plain peft.
- Зачем `apply_chat_template` именно от tokenizer'a, а не вручную.
- Как читать loss-кривую: что есть здоровый train, что overfit.
- GGUF-конвертация: почему q4_K_M по умолчанию.

**Время прогона:** ~40-60 мин на ~5k примеров за 2 эпохи.

## 0. Установка зависимостей

**Учебный нюанс:** `unsloth` — это уровень абстракции над `peft + transformers + bitsandbytes`. Он даёт ~2× speedup и ~40% меньше VRAM на T4 за счёт ручных Triton-кернелей под attention. Альтернатива — голый PEFT, работает, но медленнее.

**Почему не пиним версии вручную:** `unsloth` поставляется как единый экосистемный стек — каждый его релиз тестируется с конкретными версиями transformers/trl/datasets. Попытка зафиксировать версии самому приводит к `ResolutionImpossible` (transformers==4.51.3 несовместим с trl>=0.20). Проверено. Просто ставим `unsloth + unsloth_zoo` и даём pip разобраться.

In [ ]:
# ⚠ После выполнения → Runtime → Restart session  (НЕ "Delete runtime" — /content/*.jsonl потеряются)
#
# Стратегия: unsloth сам тянет совместимые версии transformers/trl/datasets.
# Пытаться пинить их вручную → ResolutionImpossible (проверено на практике).
# Единственное что нужно явно: unsloth_zoo (pip не тянет его как зависимость).

!pip install -q --upgrade pip
!pip install -q --upgrade unsloth unsloth_zoo

print("✓ install done — теперь Runtime → Restart session")

In [ ]:
# Запусти ПОСЛЕ Runtime → Restart session
import torch
import unsloth, unsloth_zoo, transformers, trl, datasets, peft
import numpy as np

print("✓ все библиотеки импортированы")
print()
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:           ", torch.cuda.get_device_name(0))
    print("VRAM (GB):     ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

print()
print(f"unsloth      = {unsloth.__version__}")
print(f"unsloth_zoo  = {unsloth_zoo.__version__}")
print(f"transformers = {transformers.__version__}")
print(f"trl          = {trl.__version__}")
print(f"datasets     = {datasets.__version__}")
print(f"peft         = {peft.__version__}")
print(f"torch        = {torch.__version__}")
print(f"numpy        = {np.__version__}")

# Минимальные проверки здравомыслия
assert torch.cuda.is_available(), "GPU не найден — проверь Runtime → Change runtime type → T4"
print()
print("✓ GPU есть — готов к загрузке модели")

## 1. Загрузка данных

Два варианта — выбери один:

**(A) Загрузить файлы вручную** через панель Files слева. Удобно для маленького датасета (<50 MB).

**(B) Подключить Google Drive.** Положи `train.jsonl` / `val.jsonl` в `MyDrive/text2sql_finetune/` и расшарь в Drive.

In [ ]:
# (B) Drive
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/text2sql_finetune/*.jsonl /content/
!ls -la /content/*.jsonl

In [ ]:
# Sanity-check данных
import json
with open("train.jsonl") as f:
    sample = [json.loads(line) for line in f][:3]
for ex in sample:
    msgs = ex["messages"]
    print(f"--- example ({ex.get('_meta', {}).get('task')}) ---")
    for m in msgs:
        prev = m['content'][:120].replace('\n', ' ')
        print(f"  [{m['role']:9s}] {prev}...")
    print()

## 2. Загрузка базовой модели в 4-bit

**Учебный нюанс — память T4:**
- 4B параметров в fp16 = 8 GB. Уже впритык, без места на оптимизатор.
- 4B параметров в 4-bit (NF4 quantization) = ~2.5 GB.
- LoRA-адаптеры r=16: ~50 M обучаемых параметров. В fp16 = 100 MB. AdamW state = 200 MB.
- Активации при batch=2, seq=4096: ~3 GB.
- Итого ~6 GB peak — спокойно в 16 GB T4.

**Почему 4-bit, а не 8-bit:** базовая модель ЗАМОРОЖЕНА. Её точность не критична, потому что обучается только LoRA-«дельта». В 4-bit потери качества инференса <1%.

**Если Qwen3.5-4B недоступна:** замени `model_name` на `Qwen/Qwen2.5-4B` или `Qwen/Qwen3-4B-Base` — пайплайн не зависит от конкретной версии.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096

# 3B-Instruct: ~7 GB VRAM peak на T4, полный train ~1 час.
# На нашем датасете (~800 примеров) разница в качестве с 7B небольшая.
# Если захочешь вернуться к 7B — замени на "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
# и верни per_device_train_batch_size=1, gradient_accumulation_steps=8.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
print(f"tokenizer type: {type(tokenizer).__name__}")
print(f"chat template присутствует: {bool(tokenizer.chat_template)}")

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096

# unsloth/* — зеркала популярных моделей на HuggingFace-аккаунте unsloth.
# Преимущества: pre-configured для 4-bit, совместимы с текущей версией unsloth,
# загружаются быстрее (квантизация уже встроена в конфиг модели).
# Список доступных: https://huggingface.co/unsloth
#
# Альтернатива если нужна более новая: "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" (7B, больше VRAM)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-4B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # авто: fp16 на T4 (bf16 T4 не поддерживает)
    load_in_4bit=True,
)
print(f"tokenizer type: {type(tokenizer).__name__}")
print(f"chat template присутствует: {bool(tokenizer.chat_template)}")

## 3. Wrap в LoRA

**Гиперпараметры — почему именно эти:**
- `r=16` — стандарт для 4B на специализированной задаче. Меньше r — недостаточная capacity, больше — переобучение и лишний VRAM.
- `lora_alpha=32` (= 2×r) — стандартное соотношение. Альфа отвечает за эффективный learning rate LoRA-веток.
- `target_modules` — все 7 проекций attention+MLP. Можно ограничиться только attention (`q,k,v,o`) — будет на 30% быстрее, но качество чуть ниже.
- `lora_dropout=0.05` — лёгкая регуляризация. На больших датасетах (>10k) можно 0.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # экономит VRAM ценой 20% времени
    random_state=42,
)
model.print_trainable_parameters()

## 4. Применяем chat template

**Учебный нюанс — почему `apply_chat_template`:**
Каждое семейство моделей имеет свой формат разделителей. Qwen использует ChatML:

```
<|im_start|>system
...
<|im_end|>
<|im_start|>user
...
<|im_end|>
<|im_start|>assistant
...
<|im_end|>
```

Если форматировать вручную — легко промахнуться (пропустить `\n`, перепутать токены) и модель учится мусору. `tokenizer.apply_chat_template` гарантированно даёт **тот же** строковой формат, что увидит `tokenizer.encode` на инференсе. Это та самая инвариантность train==inference.

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files={
    "train": "train.jsonl",
    "val":   "val.jsonl",
})

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

ds = raw.map(format_chat, remove_columns=["messages", "_meta"])
print(f"train: {len(ds['train'])}, val: {len(ds['val'])}")
print("--- sample formatted text (first 500 chars) ---")
print(ds["train"][0]["text"][:500])

In [ ]:
# Сколько токенов получается? Важно убедиться, что не уходим за MAX_SEQ_LEN.
import numpy as np

# Используем tokenizer(text)["input_ids"] вместо tokenizer.encode(text) —
# это работает и для PreTrainedTokenizer, и для быстрых Rust-tokenizers.
sample = ds["train"].select(range(min(500, len(ds["train"]))))
lens = [len(tokenizer(x["text"])["input_ids"]) for x in sample]

print(f"token len: p50={np.percentile(lens,50):.0f}  p90={np.percentile(lens,90):.0f}  "
      f"p99={np.percentile(lens,99):.0f}  max={max(lens)}")
truncated = sum(1 for l in lens if l > MAX_SEQ_LEN)
print(f"будут обрезаны: {truncated} / {len(lens)} ({100*truncated/len(lens):.1f}%)")
# Норма: <5% обрезается. Если больше — увеличь MAX_SEQ_LEN до 6144
# или укороти схему БД в системном промпте (убери column_stats).

from trl import SFTTrainer, SFTConfig

# Чекпоинты пишем прямо в Drive — переживут дисконнект сессии.
CKPT_DIR = "/content/drive/MyDrive/text2sql_finetune/checkpoints"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    args=SFTConfig(
        output_dir=CKPT_DIR,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch = 8
        warmup_ratio=0.05,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,                       # A100/H100: bf16 точнее fp16 и быстрее
        logging_steps=10,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=5,
        eval_strategy="steps",
        eval_steps=100,
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field="text",
        packing=True,
        report_to="none",
        seed=42,
    ),
)
print(f"Чекпоинты будут сохраняться в: {CKPT_DIR}")

In [ ]:
# Smoke-train: 20 шагов на маленьком сабсэмпле.
# Цель: убедиться что loss падает (а не NaN / взрывается) до запуска полного прогона.
import copy

smoke_args = copy.deepcopy(trainer.args)
smoke_args.max_steps = 20
smoke_args.eval_strategy = "no"   # ← меняем ДО SFTTrainer.__init__
smoke_args.save_strategy = "no"   #   иначе он проверяет eval_dataset и падает

smoke_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"].select(range(min(64, len(ds["train"])))),
    args=smoke_args,
)
smoke_trainer.train()
# Ожидаем: loss ~1.5-3.0 в начале, плавно падает к ~0.5-1.5 за 20 шагов.
# NaN или рост loss → проблема с chat-template или dtype.

In [ ]:
import shutil
from pathlib import Path

FORCE_FRESH = False  # True → игнорировать старые чекпоинты и стартовать с нуля
                     # (например, если сменил fp16↔bf16 или поменял модель)

ckpt_dir = Path(CKPT_DIR)
last_ckpt = None

if not FORCE_FRESH and ckpt_dir.exists():
    ckpts = sorted(
        [d for d in ckpt_dir.iterdir() if d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[1])
    )
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f"↻ Resuming from {last_ckpt}")
    else:
        print("▶ Чекпоинтов нет — старт с нуля")
else:
    if FORCE_FRESH:
        # Удаляем старые чекпоинты чтобы не было конфликта dtype/scaler
        if ckpt_dir.exists():
            shutil.rmtree(ckpt_dir)
            print("🗑 Старые чекпоинты удалены (FORCE_FRESH=True)")
    print("▶ Старт с нуля")

trainer.train(resume_from_checkpoint=last_ckpt)

In [ ]:
# Полный прогон
trainer.train()

In [ ]:
# Сохраняем LoRA-адаптер (~150 MB, маленький)
model.save_pretrained("outputs/qwen35-4b-text2sql-lora/adapter")
tokenizer.save_pretrained("outputs/qwen35-4b-text2sql-lora/adapter")
!ls -la outputs/qwen35-4b-text2sql-lora/adapter/
!du -sh outputs/qwen35-4b-text2sql-lora/adapter/

## 6. Quick eval — посмотреть, как модель отвечает

Перед тем как мерджить и конвертировать — ручная проверка на 2-3 примерах из val. Если модель отвечает мусором или не следует формату — что-то не так с chat template / training format.

In [ ]:
FastLanguageModel.for_inference(model)  # переключает unsloth в режим инференса (ускорение)

import json
raw_val = [json.loads(l) for l in open("val.jsonl")][:3]

for ex in raw_val:
    msgs = ex["messages"][:2]  # без assistant
    inputs = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to("cuda")
    out = model.generate(inputs, max_new_tokens=256, do_sample=False, temperature=0.0)
    pred = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    gold = ex["messages"][2]["content"]
    print(f"--- {ex.get('_meta', {})} ---")
    print(f"PRED: {pred[:300]}")
    print(f"GOLD: {gold[:300]}\n")

## 7. Merge LoRA → HF safetensors → GGUF q4_K_M

**Учебный нюанс — формат GGUF:**
- `q4_K_M` — 4-bit с K-quants и Medium-блоками. Лучший trade-off для 4B: ~3 GB файл, потеря качества <1% относительно fp16.
- `q5_K_M` — 5-bit, ~3.5 GB, потери почти 0. Если место не критично — бери.
- `q8_0` — 8-bit, ~4.5 GB. Эталон, минимальные потери. Лучше для финальной публикации.
- `q3_K_S` — 3-bit, ~2 GB. Заметно деградирует на сложных SQL. Не бери для production.

Конвертация ~5 минут на 4B.

In [ ]:
# unsloth helper делает merge + GGUF в одной операции.
# ВАЖНО: unsloth добавляет суффикс _gguf к имени папки автоматически.
# Т.е. при имени "qwen35-text2sql-lora" файл окажется в qwen35-text2sql-lora_gguf/
model.save_pretrained_gguf(
    "qwen35-text2sql-lora",
    tokenizer,
    quantization_method="q4_k_m",
)
# Ищем GGUF независимо от того, как unsloth назвал папку
import glob as _glob
gguf_found = _glob.glob("/content/**/*.gguf", recursive=True)
print("GGUF файлы:", gguf_found)


In [ ]:
import shutil, glob
from pathlib import Path

# Ищем GGUF только в локальном хранилище Colab (исключаем /content/drive/ — там уже Drive)
gguf_files = [
    f for f in glob.glob("/content/**/*.gguf", recursive=True)
    if "/content/drive/" not in f
]
print("Найдено локально:", gguf_files)

# Берём только Q4_K_M (не BF16 — он ~15 GB и не нужен для LM Studio)
gguf_q4 = [f for f in gguf_files if "Q4_K_M" in f or "q4_k_m" in f]
print("К копированию (Q4_K_M):", gguf_q4)

if not gguf_q4:
    print("❌ Q4_K_M файл не найден. Все GGUF:", gguf_files)
else:
    dst_dir = Path("/content/drive/MyDrive/text2sql_finetune/")
    dst_dir.mkdir(parents=True, exist_ok=True)

    for src in gguf_q4:
        src_path = Path(src)
        dst = dst_dir / src_path.name
        if dst.exists():
            print(f"⚠ уже есть в Drive: {dst.name} — пропускаю")
            continue
        print(f"Копирую {src_path.name} ({src_path.stat().st_size/1e9:.2f} GB)...")
        shutil.copy(src, dst)
        print(f"✓ {dst}")


## 8. Что дальше

1. Скачай GGUF локально на Mac.
2. Положи в `~/.lmstudio/models/local/qwen35-text2sql/qwen35-text2sql.q4_k_m.gguf` (создай папку — LM Studio её увидит).
3. В LM Studio: открой модель → Local Server → Start.
4. В `.env` проекта переключи:
   ```
   LLM_BASE_URL=http://localhost:1234/v1
   LLM_API_KEY=lm-studio
   LLM_MODEL_NAME=qwen35-text2sql
   ```
5. `./scripts/run_ablation.sh feat/lora_finetune` — eval на bird_small + ambrosia_small.

См. `scripts/lm_studio_smoke_test.py` для проверки локального API.